# 999: Combine the Salary Datasets into One Table

This notebook merges the outputs of notebooks 01-03 into **one salary-feature table** with the project's unit, **one row = one player + season + team**. It keeps **only players under contract for the next season** (section 6), the population our research question covers. That table is what gets joined with the other datasets' tables (stats, team results, ...) before XGBoost.

| Input (`data-files/02-processed/`) | From | Row = |
|---|---|---|
| `llimllib_player_salary_features.csv` | 02 | player + season + team (already includes the team payroll and tax features, which use 01's thresholds) |
| `espn_contract_features.csv` | 03 | player + season (team = end-of-season team) |

**Output:** `data-files/02-processed/salary_features_combined.csv`

*Run order: `01` → `02` → `03` → `999`.*

## Setup

In [1]:
import pandas as pd

from nba_cleaning_utils import PROCESSED, FIRST_SEASON, LAST_SEASON

salary = pd.read_csv(PROCESSED / 'llimllib_player_salary_features.csv', dtype={'nba_player_id': str})
contracts = pd.read_csv(PROCESSED / 'espn_contract_features.csv', dtype={'espn_athlete_id': str})

print('llimllib salary rows:', salary.shape)
print('ESPN contract rows:  ', contracts.shape)

llimllib salary rows: (6767, 24)
ESPN contract rows:   (5534, 9)


## 1. Choose the base table

The llimllib table is the base:

- Every row has an NBA `player_id`. That's the id NBA stats data uses, which makes it the most reliable key for the joins with the other datasets.
- It already has the project's unit (player + season + team). For a player paid by two teams in one season (traded or waived), there's one row per team.

ESPN's contract columns get **left-joined** onto it, so every llimllib row is kept and a player with no ESPN match gets `NaN` contract features. XGBoost handles `NaN` natively.

## 2. Join keys

1. **`season + team + name_key`**: the strict match. Both sources record the player's end-of-season team.
2. **`season + name_key`**: a fallback for rows the strict match misses, used **only if** that name appears exactly once in the ESPN data for that season. It catches cases where the two sources disagree about a player's team, e.g. a player waived late in the season.

Before joining, make sure each key is unique on the ESPN side, so the join can't create duplicate rows.

In [2]:
strict_key = ['season', 'team', 'name_key']
loose_key = ['season', 'name_key']

print('ESPN rows that share a strict key:', contracts.duplicated(strict_key).sum())
print('ESPN rows that share a season + name key:', contracts.duplicated(loose_key, keep=False).sum())
print('llimllib rows that share season + team + player id:', salary.duplicated(['season', 'team', 'nba_player_id']).sum())

ESPN rows that share a strict key: 0
ESPN rows that share a season + name key: 0
llimllib rows that share season + team + player id: 0


## 3. Merge

In [3]:
contract_cols = ['espn_athlete_id', 'espn_salary', 'years_remaining', 'expiring_contract', 'min_salary_contract']

# Pass 1: strict match
merged = salary.merge(contracts[strict_key + contract_cols], on=strict_key, how='left', validate='many_to_one')
merged['contract_match'] = merged['espn_athlete_id'].notna().map({True: 'season+team+name', False: None})

# Pass 2: season + name fallback, only for ESPN names that are unique within the season
unique_names = contracts[~contracts.duplicated(loose_key, keep=False)]
missing = merged['espn_athlete_id'].isna()
fallback = merged.loc[missing, loose_key].merge(unique_names[loose_key + contract_cols], on=loose_key, how='left')
fallback.index = merged.index[missing]

# Each ESPN contract (season + athlete) may be attached to at most one row:
#   - skip contracts already used by a strict match (e.g. the player's other row that season)
#   - if two unmatched rows want the same contract, only the first gets it
contract_id = lambda d: d['season'].astype(str) + '|' + d['espn_athlete_id'].astype(str)
already_used = set(contract_id(merged.loc[~missing]))
candidates = fallback[fallback['espn_athlete_id'].notna()]
candidates = candidates[~contract_id(candidates).isin(already_used)]
candidates = candidates[~contract_id(candidates).duplicated()]

merged.loc[candidates.index, contract_cols] = candidates[contract_cols]
merged.loc[candidates.index, 'contract_match'] = 'season+name'

print('rows:', len(merged), '(same as the llimllib base:', len(merged) == len(salary), ')')
merged['contract_match'].value_counts(dropna=False)

rows: 6767 (same as the llimllib base: True )


contract_match
season+team+name    5426
NaN                 1256
season+name           85
Name: count, dtype: int64

## 4. Check the merge

In [4]:
coverage = merged.assign(matched=merged['espn_athlete_id'].notna()).groupby('season')['matched'].mean().mul(100).round(1)
coverage.rename('% of rows with ESPN contract data').to_frame().T

season,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
% of rows with ESPN contract data,84.7,81.7,74.4,85.2,76.6,86.4,70.9,85.1,83.5,85.8,83.6


In [5]:
# Were all ESPN contracts used? (The ones left over are players llimllib doesn't list for that season.)
used_ids = set(contract_id(merged.dropna(subset=['espn_athlete_id'])))
unused = contracts[~contract_id(contracts).isin(used_ids)]
print('ESPN contracts not attached to any row:', len(unused), f'({len(unused) / len(contracts):.1%})')
unused.sample(min(8, len(unused)), random_state=0)[['season', 'team', 'player_name', 'espn_salary']]

ESPN contracts not attached to any row: 23 (0.4%)


,season,team,player_name,espn_salary
1707,2019,MIN,Cameron Reynolds,108953.0
1690,2019,MIL,Jaylen Morris,1378242.0
3976,2024,CHA,Nate Mensah,0.0
2143,2020,MIA,Daryl Macon,0.0
2752,2021,PHI,Dakota Mathias,0.0
582,2017,DAL,Pierre Deshawn Jackson,31969.0
2038,2020,GSW,Omari Rasulala Spellman,1897800.0
4479,2025,BKN,Yongxi Cui,0.0


In [6]:
# Sanity check: matched rows should have (nearly) the same salary in both sources
m = merged[merged['espn_salary'].notna() & (merged['cap_salary'] > 0)]
ratio = m['espn_salary'] / m['cap_salary']
print('matched rows:', len(m))
print('ESPN salary within 5% of llimllib salary:', f'{ratio.between(0.95, 1.05).mean():.1%}')
m.assign(ratio=ratio).loc[~ratio.between(0.5, 2), ['season', 'team', 'player_name', 'cap_salary', 'espn_salary']].head(8)

matched rows: 5021
ESPN salary within 5% of llimllib salary: 86.1%


,season,team,player_name,cap_salary,espn_salary
14,2016,ATL,Kris Humphries,388025,1000000.0
32,2016,BKN,Sean Kilpatrick,320594,134215.0
33,2016,BKN,Henry Sims,161622,947276.0
102,2016,CLE,Dahntay Jones,8819,0.0
139,2016,DEN,Kostas Papanikolaou,350000,800321.0
195,2016,HOU,Chuck Hayes,83583,947276.0
227,2016,LAC,Jeff Ayres,317232,111444.0
264,2016,MEM,Jordan Farmar,191456,0.0


In [7]:
# Rows with no contract match: who are they?
no_match = merged[merged['espn_athlete_id'].isna()]
print('rows without ESPN contract data:', len(no_match))
print('...of which are two-way contracts:', no_match['is_two_way'].sum())
print('...median salary:', f"${no_match['cap_salary'].median():,.0f}")

rows without ESPN contract data: 1256
...of which are two-way contracts: 219
...median salary: $179,058


**What we found:** Rows without an ESPN contract are mostly **two-way players, waived players' dead-money rows, and short-term signings**. ESPN either has no contract on file for them, or they didn't play for the team that still pays them. A large share of matched rows have nearly the same salary in both sources, which confirms the name join is attaching the right players. The big gaps are almost all **short contracts** (10-day deals, mid-season signings), where one source records the pro-rated amount actually paid and the other the full-season figure. Our features use the llimllib salary (`cap_salary`), so this only matters as a check that the join is right.

## 5. Final columns

Keep the keys, the features, and a few columns that help the next joins. Drop the columns that were only used for checking (`espn_salary`, `contract_match`).

`min_salary_over_cap` is the version of the minimum-salary feature you described: the player was signed with the minimum exception **and** the team's payroll was over the cap.

In [8]:
merged['min_salary_over_cap'] = (merged['min_salary_contract'].astype('boolean') & merged['over_cap'].astype('boolean'))

final_cols = [
    # keys
    'season', 'team', 'nba_player_id', 'player_name', 'name_key', 'bbref_id', 'espn_athlete_id',
    # player salary
    'cap_salary', 'is_two_way', 'salary_pct_of_cap',
    # contract (ESPN)
    'years_remaining', 'expiring_contract', 'min_salary_contract', 'min_salary_over_cap',
    # team payroll + luxury-tax markers
    'team_payroll', 'payroll_pct_of_cap', 'tax_tier',
    'over_cap', 'over_tax', 'over_first_apron', 'over_second_apron',
    # dollars above (+) / below (-) each threshold
    'dollars_vs_cap', 'dollars_vs_tax', 'dollars_vs_first_apron', 'dollars_vs_second_apron',
    'dollars_vs_cap_pct_cap', 'dollars_vs_tax_pct_cap', 'dollars_vs_first_apron_pct_cap', 'dollars_vs_second_apron_pct_cap',
]
salary_features = (merged[final_cols]
                   .astype({'years_remaining': 'Int64', 'expiring_contract': 'boolean', 'min_salary_contract': 'boolean',
                            'over_cap': 'boolean', 'over_tax': 'boolean', 'over_first_apron': 'boolean',
                            'over_second_apron': 'boolean'})
                   .sort_values(['season', 'team', 'cap_salary'], ascending=[True, True, False])
                   .reset_index(drop=True))

assert salary_features['nba_player_id'].notna().all()
assert not salary_features.duplicated(['season', 'team', 'nba_player_id']).any()
assert salary_features['season'].between(FIRST_SEASON, LAST_SEASON).all()

print(salary_features.shape)
salary_features.head()

(6767, 29)


,season,team,nba_player_id,player_name,name_key,bbref_id,espn_athlete_id,cap_salary,is_two_way,salary_pct_of_cap,...,over_first_apron,over_second_apron,dollars_vs_cap,dollars_vs_tax,dollars_vs_first_apron,dollars_vs_second_apron,dollars_vs_cap_pct_cap,dollars_vs_tax_pct_cap,dollars_vs_first_apron_pct_cap,dollars_vs_second_apron_pct_cap
0,2016,ATL,200794,Paul Millsap,paul millsap,millspa01,3015,19000000,False,27.142857,...,False,<NA>,1453126.0,-13286874.0,-17286874.0,NaN,2.075894,-18.981249,-24.695534,NaN
1,2016,ATL,201143,Al Horford,al horford,horfoal01,3213,12000000,False,17.142857,...,False,<NA>,1453126.0,-13286874.0,-17286874.0,NaN,2.075894,-18.981249,-24.695534,NaN
2,2016,ATL,201168,Tiago Splitter,tiago splitter,splitti01,3233,8500000,False,12.142857,...,False,<NA>,1453126.0,-13286874.0,-17286874.0,NaN,2.075894,-18.981249,-24.695534,NaN
3,2016,ATL,201952,Jeff Teague,jeff teague,teaguje01,4015,8000000,False,11.428571,...,False,<NA>,1453126.0,-13286874.0,-17286874.0,NaN,2.075894,-18.981249,-24.695534,NaN
4,2016,ATL,2594,Kyle Korver,kyle korver,korveky01,2011,5746479,False,8.209256,...,False,<NA>,1453126.0,-13286874.0,-17286874.0,NaN,2.075894,-18.981249,-24.695534,NaN


In [9]:
salary_features.isna().sum().loc[lambda s: s > 0].rename('missing values').to_frame()

,missing values
espn_athlete_id,1256
years_remaining,1355
expiring_contract,1355
min_salary_contract,1256
min_salary_over_cap,1058
over_second_apron,4834
dollars_vs_second_apron,4834
dollars_vs_second_apron_pct_cap,4834


**Expected missing values (before the section 6 filter):**

- `espn_athlete_id`, `years_remaining`, `expiring_contract`, `min_salary_contract`, `min_salary_over_cap`: rows with no ESPN contract (section 4), plus the ~100 rows where ESPN's `years_remaining` was impossible (notebook 03).
  `min_salary_over_cap` has fewer blanks than `min_salary_contract`, because a player on a team **under** the cap is `False` whether or not his contract is known.
- `over_second_apron`, `dollars_vs_second_apron*`: every season before 2023-24, when the second apron didn't exist.

Leave these as `NaN`. XGBoost learns which way to send missing values at each split.

In [10]:
salary_features.describe().T[['count', 'mean', 'min', '50%', 'max']]

,count,mean,min,50%,max
season,6767.0,2021.149549,2016.0,2021.0,2026.0
cap_salary,6767.0,6438219.040343,0.0,2389641.0,59606817.0
salary_pct_of_cap,6767.0,5.622211,0.0,2.136998,40.725324
years_remaining,5412.0,2.236881,1.0,2.0,6.0
team_payroll,6767.0,132628024.873356,53540542.0,128994563.0,214371340.0
payroll_pct_of_cap,6767.0,114.827428,70.749557,116.719812,158.051091
tax_tier,6767.0,1.247673,0.0,1.0,4.0
dollars_vs_cap,6767.0,17929725.774789,-32881593.0,18053456.0,73783340.0
dollars_vs_tax,6767.0,-6515919.119551,-57073593.0,-5511804.0,43557340.0
dollars_vs_first_apron,6767.0,-12768879.51559,-63469593.0,-11829888.0,36239340.0


## 6. Keep only players under contract for the next season

Our research question only covers players whose contract continues into the next season. When a contract ends this season (`years_remaining = 1`), the player becomes a free agent in July and can't be traded in the offseason except through a sign-and-trade. Keeping these players would mostly teach the model who is **ineligible** to be traded rather than who teams **choose** to trade.

- **Keep:** rows with `expiring_contract == False`, i.e. `years_remaining >= 2`.
- **Drop:** expiring contracts, and rows where the contract length is unknown (no ESPN contract, or ESPN's impossible `years_remaining = 0` from notebook 03). For those we can't tell whether the player is under contract.
- **Drop the `expiring_contract` column.** After the filter it is `False` for every row, so it carries no information for the model.

In [11]:
status = (salary_features['expiring_contract']
          .map({True: 'expiring', False: 'under contract'})
          .fillna('unknown contract length'))
print(status.value_counts().to_string())
print()

under_contract = (salary_features[salary_features['expiring_contract'].eq(False).fillna(False)]
                  .drop(columns='expiring_contract')
                  .reset_index(drop=True))

assert (under_contract['years_remaining'] >= 2).all()
print('rows kept:', under_contract.shape)
status.groupby(salary_features['season']).value_counts().unstack()

expiring_contract
under contract             3478
expiring                   1934
unknown contract length    1355

rows kept: (3478, 28)


expiring_contract,expiring,under contract,unknown contract length
season,,,
2016,142,315,93
2017,133,334,117
2018,180,271,162
2019,201,270,115
2020,137,289,154
2021,185,308,94
2022,172,331,214
2023,190,335,92
2024,201,333,113


In [12]:
# Missing values left after the filter (only the second-apron columns should remain)
under_contract.isna().sum().loc[lambda s: s > 0].rename('missing values').to_frame()

,missing values
over_second_apron,2453
dollars_vs_second_apron,2453
dollars_vs_second_apron_pct_cap,2453


**What we found:** 3,478 of the 6,767 player-season-team rows (51%) are players under contract for the next season. We dropped 1,934 expiring contracts and 1,355 rows with unknown contract length. Each season keeps roughly 270-355 players. The only missing values left are the second-apron columns before 2023-24, which XGBoost handles natively.

Dropping the unknown rows mostly removes two-way players, dead-money rows, and short-term signings (section 4), but it also removes the ~100 rows where ESPN reported an impossible `years_remaining = 0`, some of which are players clearly under contract (notebook 03). That is a small, known gap in the sample.

## 7. Save

In [13]:
out = PROCESSED / 'salary_features_combined.csv'
under_contract.to_csv(out, index=False)
print('saved', under_contract.shape, 'to', out)

saved (3478, 28) to ..\..\data-files\02-processed\salary_features_combined.csv


**Output:** `data-files/02-processed/salary_features_combined.csv`, one row per player + season + team, **only for players under contract for the next season**.

**How to join it with the other datasets**

- **Use this table as the base** and left-join the other datasets onto it. The stats tables cover every player, so joining the other way would bring back the expiring-contract players we filtered out.

- **Preferred key:** `season` (end year, 2026 = 2025-26) + `nba_player_id` (+ `team`). This works directly with any NBA.com / stats.nba.com-based table.
- **Name-based tables:** `season + team + name_key`. Build `name_key` with `normalize_name()` from `nba_cleaning_utils.py` so the keys match exactly.
- A player with two rows in one season (e.g., traded, or waived with dead money still on the old team) keeps both rows. When joining to per-team stats, the row for a team he never played for (dead money only) will have no stats match and can be dropped then.